<a href="https://colab.research.google.com/github/ddsntc1/Visual_information_extraction/blob/main/model_training_main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!huggingface-cli login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) n
Token is valid (permission: write).
The token `p1` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `p1`


In [ ]:
!pip install --upgrade pip
!pip install transformers datasets seqeval pillow rich faiss-gpu sentence_transformers
!pip install evaluate
!pip install -U accelerate
!sudo apt install tesseract-ocr
!pip install --no-deps pytesseract
!git lfs install
!git clone https://huggingface.co/microsoft/layoutlmv3-base

In [ ]:
from datasets import load_dataset, Features, Sequence, ClassLabel, Value, Array2D, Array3D
from datasets.features import ClassLabel
from transformers import AutoProcessor
from PIL import Image
import torch
import numpy as np

dataset = load_dataset("Dongwookss/SROIE_lb1")
processor = AutoProcessor.from_pretrained("microsoft/layoutlmv3-base", apply_ocr=False)

features = dataset["train"].features
column_names = dataset["train"].column_names
image_column_name = "image"
text_column_name = "words"
boxes_column_name = "bbox"
label_column_name = "label"

def get_label_list(labels):
    unique_labels = set()
    for label in labels:
        unique_labels = unique_labels | set(label)
    label_list = list(unique_labels)
    label_list.sort()
    return label_list

if isinstance(features[label_column_name].feature, ClassLabel):
    label_list = features[label_column_name].feature.names
    id2label = {k: v for k,v in enumerate(label_list)}
    label2id = {v: k for k,v in enumerate(label_list)}
else:
    label_list = get_label_list(dataset["train"][label_column_name])
    id2label = {k: v for k,v in enumerate(label_list)}
    label2id = {v: k for k,v in enumerate(label_list)}
num_labels = len(label_list)
print(id2label)
print(label_list)

def prepare_examples(examples, window_size=384, stride=192):
    images = examples[image_column_name]
    words = examples[text_column_name]
    boxes = examples[boxes_column_name]
    word_labels = examples[label_column_name]

    # 결과를 저장할 리스트들
    all_encoded = {
        'pixel_values': [],
        'input_ids': [],
        'attention_mask': [],
        'bbox': [],
        'labels': []
    }

    for idx in range(len(images)):
        # 이미지 전처리
        image = images[idx]
        if not isinstance(image, Image.Image):
            image = Image.open(image)
        if image.mode != "RGB":
            image = image.convert("RGB")

        current_words = words[idx]
        current_boxes = boxes[idx]
        current_labels = word_labels[idx]

        # 문서가 긴 경우 여러 윈도우로 처리
        if len(current_words) > window_size:
            for start_idx in range(0, len(current_words), stride):
                end_idx = min(start_idx + window_size, len(current_words))

                # 마지막 윈도우는 끝에서부터 역으로 계산
                if end_idx == len(current_words):
                    start_idx = max(0, end_idx - window_size)

                # 현재 윈도우의 데이터 추출
                window_words = current_words[start_idx:end_idx]
                window_boxes = current_boxes[start_idx:end_idx]
                window_labels = current_labels[start_idx:end_idx]

                # processor로 처리
                encoding = processor(
                    image,
                    window_words,
                    boxes=window_boxes,
                    word_labels=window_labels,
                    truncation=True,
                    padding="max_length",
                    return_tensors="pt"
                )

                # 결과 저장
                for key in all_encoded.keys():
                    if isinstance(encoding[key], torch.Tensor):
                        all_encoded[key].append(encoding[key][0].numpy())
                    else:
                        all_encoded[key].append(encoding[key])
        else:
            # 짧은 문서는 그대로 처리
            encoding = processor(
                image,
                current_words,
                boxes=current_boxes,
                word_labels=current_labels,
                truncation=True,
                padding="max_length",
                return_tensors="pt"
            )

            # 결과 저장
            for key in all_encoded.keys():
                if isinstance(encoding[key], torch.Tensor):
                    all_encoded[key].append(encoding[key][0].numpy())
                else:
                    all_encoded[key].append(encoding[key])

    # numpy 배열을 리스트로 변환
    result = {}
    for key, value in all_encoded.items():
        if isinstance(value[0], np.ndarray):
            result[key] = value
        else:
            result[key] = value

    return result

features = Features({
    'pixel_values': Array3D(dtype="float32", shape=(3, 224, 224)),
    'input_ids': Sequence(feature=Value(dtype='int64')),
    'attention_mask': Sequence(Value(dtype='int64')),
    'bbox': Array2D(dtype="int64", shape=(512, 4)),
    'labels': Sequence(feature=Value(dtype='int64')),
})

train_dataset = dataset["train"].map(
    prepare_examples,
    batched=True,
    remove_columns=column_names,
    features=features,
)
eval_dataset = dataset["test"].map(
    prepare_examples,
    batched=True,
    remove_columns=column_names,
    features=features,
)

README.md:   0%|          | 0.00/699 [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/319M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/183M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/626 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/347 [00:00<?, ? examples/s]

...

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

{0: 'S-COMPANY', 1: 'S-DATE', 2: 'S-ADDRESS', 3: 'S-TOTAL', 4: 'O'}
['S-COMPANY', 'S-DATE', 'S-ADDRESS', 'S-TOTAL', 'O']


Map:   0%|          | 0/626 [00:00<?, ? examples/s]

Map:   0%|          | 0/347 [00:00<?, ? examples/s]

In [ ]:
from evaluate import load

metric = load("seqeval")

return_entity_level_metrics = False

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # stride 크기를 고려한 예측값 통합
    true_predictions = []
    true_labels = []

    current_pred = []
    current_label = []
    last_end = 0

    for pred, label in zip(predictions, labels):
        valid_indices = label != -100
        current_pred.extend([label_list[p] for p, valid in zip(pred, valid_indices) if valid])
        current_label.extend([label_list[l] for l, valid in zip(label, valid_indices) if valid])
        if len(current_pred) >= 192:  # stride 크기
            true_predictions.append(current_pred[last_end:])
            true_labels.append(current_label[last_end:])
            last_end = 192  # stride 크기

    if current_pred[last_end:]:
        true_predictions.append(current_pred[last_end:])
        true_labels.append(current_label[last_end:])

    results = metric.compute(predictions=true_predictions, references=true_labels)

    if return_entity_level_metrics:
        final_results = {}
        for key, value in results.items():
            if isinstance(value, dict):
                for n, v in value.items():
                    final_results[f"{key}_{n}"] = v
            else:
                final_results[key] = value
        return final_results
    else:
        return {
            "precision": results["overall_precision"],
            "recall": results["overall_recall"],
            "f1": results["overall_f1"],
            "accuracy": results["overall_accuracy"],
        }

In [ ]:
from transformers import LayoutLMv3ForTokenClassification, TrainingArguments, Trainer ,EarlyStoppingCallback
from transformers.data.data_collator import default_data_collator

model = LayoutLMv3ForTokenClassification.from_pretrained("microsoft/layoutlmv3-base", # layoutlmv3-base
                                                         id2label=id2label,
                                                         label2id=label2id)

# 학습 설정 수정

training_args = TrainingArguments(
    output_dir="test",
    max_steps=1500,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    learning_rate=2e-5,
    evaluation_strategy="steps",
    eval_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    gradient_accumulation_steps=4  # 메모리 부족시 활용
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=processor,
    data_collator=default_data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]

Some weights of LayoutLMv3ForTokenClassification were not initialized from the model checkpoint at microsoft/layoutlmv3-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
<ipython-input-7-d91adeb9eccf>:23: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
max_steps is given, it will override any value given in num_train_epochs
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` 

<IPython.core.display.Javascript object>

wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


/usr/local/lib/python3.10/dist-packages/transformers/modeling_utils.py:1161: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(


Step,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
100,No log,0.062756,0.931546,0.947225,0.939321,0.981596
200,No log,0.054161,0.931870,0.963385,0.947365,0.984127
300,No log,0.054317,0.930237,0.964487,0.947052,0.983781
400,No log,0.055134,0.948607,0.957941,0.953251,0.985896
500,0.064200,0.055304,0.957826,0.959947,0.958885,0.987685
600,0.064200,0.054805,0.958688,0.960758,0.959722,0.987855
700,0.064200,0.061992,0.955959,0.963159,0.959545,0.987769
800,0.064200,0.068064,0.952974,0.965010,0.958954,0.987617
900,0.064200,0.071902,0.948357,0.965531,0.956867,0.986926
1000,0.011000,0.072447,0.952831,0.964279,0.958521,0.987467


/usr/local/lib/python3.10/dist-packages/transformers/modeling_utils.py:1161: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(


TrainOutput(global_step=1000, training_loss=0.037594826698303226, metrics={'train_runtime': 2929.1954, 'train_samples_per_second': 2.731, 'train_steps_per_second': 0.341, 'total_flos': 2108804947968000.0, 'train_loss': 0.037594826698303226, 'epoch': 12.779552715654953})

In [ ]:
trainer.evaluate()

/usr/local/lib/python3.10/dist-packages/transformers/modeling_utils.py:1161: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(


{'eval_loss': 0.05530431494116783,
 'eval_precision': 0.957825791688338,
 'eval_recall': 0.9599470478911314,
 'eval_f1': 0.9588852466247527,
 'eval_accuracy': 0.9876852720592767,
 'eval_runtime': 172.0195,
 'eval_samples_per_second': 2.017,
 'eval_steps_per_second': 1.012,
 'epoch': 12.779552715654953}

### Inference

In [ ]:
opdataset = load_dataset("Dongwookss/SROIE_op")

import pandas as pd
from tqdm import tqdm

model = LayoutLMv3ForTokenClassification.from_pretrained("test/checkpoint-1000")
processor = AutoProcessor.from_pretrained("test/checkpoint-1000", apply_ocr=False)

def safe_process_window(image, words, bbox, file_name, model, processor, window_size=384):
    try:
        encoding = processor(
            image,
            words,
            boxes=bbox,
            return_tensors="pt",
            max_length=window_size,
            truncation=True
        )

        # word_ids 확인
        word_ids = encoding.word_ids()
        if word_ids is None:
            print(f"Warning: word_ids is None for window in {file_name}")
            return None

        outputs = model(**encoding)
        predictions = outputs.logits.argmax(-1).squeeze().tolist()
        if isinstance(predictions, int):
            predictions = [predictions]

        return word_ids, predictions

    except Exception as e:
        print(f"Error in window processing for {file_name}: {str(e)}")
        return None

def process_with_sliding_window(example, model, processor, window_size=384, stride=192):
    words = example['words']
    bbox = example['bbox']
    image = example["image"]
    file_name = example['file_name']

    if image.mode != "RGB":
        image = image.convert("RGB")

    # 전체 결과를 저장할 리스트
    all_results = [None] * len(words)

    # 첫 번째 윈도우 처리
    first_window = safe_process_window(image, words, bbox, file_name, model, processor, window_size)

    if first_window is not None:
        word_ids, predictions = first_window

        # 첫 번째 윈도우의 결과 저장
        current_word_idx = -1
        for token_idx, word_idx in enumerate(word_ids):
            if word_idx is None:
                continue

            if word_idx != current_word_idx:
                current_word_idx = word_idx
                if word_idx < len(words):
                    all_results[word_idx] = {
                        'word': words[word_idx],
                        'predicted_label': id2label.get(predictions[token_idx], str(predictions[token_idx])),
                        'filename': file_name
                    }

    # 처리되지 않은 단어 확인
    unprocessed = [i for i, r in enumerate(all_results) if r is None]

    # 처리되지 않은 단어가 있으면 두 번째 윈도우 처리
    if unprocessed:
        start_idx = max(0, unprocessed[0] - stride)
        remaining_words = words[start_idx:]
        remaining_bbox = bbox[start_idx:]

        second_window = safe_process_window(image, remaining_words, remaining_bbox,
                                         file_name, model, processor, window_size)

        if second_window is not None:
            word_ids, predictions = second_window

            # 두 번째 윈도우의 결과 저장
            current_word_idx = -1
            for token_idx, word_idx in enumerate(word_ids):
                if word_idx is None:
                    continue

                if word_idx != current_word_idx:
                    current_word_idx = word_idx
                    if word_idx < len(remaining_words):
                        original_idx = start_idx + word_idx
                        if original_idx < len(words) and all_results[original_idx] is None:
                            all_results[original_idx] = {
                                'word': words[original_idx],
                                'predicted_label': id2label.get(predictions[token_idx], str(predictions[token_idx])),
                                'filename': file_name
                            }

    # 미처리된 단어들에 기본 레이블 할당
    final_results = []
    for i, result in enumerate(all_results):
        if result is None:
            final_results.append({
                'word': words[i],
                'predicted_label': 'O',
                'filename': file_name
            })
        else:
            final_results.append(result)
    return final_results

# 전체 문서 처리
all_results = []
try:
    for example in tqdm(opdataset["op"], desc="Processing images"):
        results = process_with_sliding_window(example, model, processor)
        all_results.extend(results)
except Exception as e:
    print(f"\nUnexpected error during processing: {str(e)}")

# 결과 저장
df = pd.DataFrame(all_results)
output_path = 'inference_results.csv'
df.to_csv(output_path, index=False, header=False, encoding='utf-8')

# 최종 통계
print(f"\n총 처리된 단어 수: {len(df)}")
print("\n레이블 분포:")
print(df['predicted_label'].value_counts())

# 원본 단어 수와 비교
total_original_words = sum(len(example['words']) for example in opdataset["op"])
print(f"\n원본 총 단어 수: {total_original_words}")
print(f"처리된 총 단어 수: {len(df)}")
if total_original_words != len(df):
    print("Warning: 원본과 처리된 단어 수가 다릅니다!")

Processing images: 100%|██████████| 347/347 [02:16<00:00,  2.54it/s]



총 처리된 단어 수: 43786

레이블 분포:
predicted_label
O            37538
S-ADDRESS     3975
S-COMPANY     1633
S-DATE         420
S-TOTAL        220
Name: count, dtype: int64

원본 총 단어 수: 43786
처리된 총 단어 수: 43786


### 모델 결과 저장 - huggingface hub 사용

In [ ]:
from huggingface_hub import HfApi
import os

def upload_folder_to_huggingface(
    folder_path: str,
    repo_id: str,
    token: str = None,
    repo_type: str = "model",
    commit_message: str = "Upload model files"
):
    api = HfApi()
    try:
        api.create_repo(
            repo_id=repo_id,
            repo_type=repo_type,
            private=False,
            token=token
        )
    except Exception as e:
        print(f"레포지토리가 이미 존재하거나 생성 중 오류 발생: {e}")
    api.upload_folder(
        folder_path=folder_path,
        repo_id=repo_id,
        repo_type=repo_type,
        token=token,
        commit_message=commit_message
    )

rng_state.pth:   0%|          | 0.00/14.2k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]

optimizer.pt:   0%|          | 0.00/1.00G [00:00<?, ?B/s]

scheduler.pt:   0%|          | 0.00/1.06k [00:00<?, ?B/s]

training_args.bin:   0%|          | 0.00/5.24k [00:00<?, ?B/s]

Upload 5 LFS files:   0%|          | 0/5 [00:00<?, ?it/s]

모든 파일이 성공적으로 업로드되었습니다: https://huggingface.co/Dongwookss/vie_task_v6_data


In [ ]:
# 환경 변수에서 토큰을 가져오거나 직접 지정
HF_TOKEN =

# 업로드할 폴더와 레포지토리 정보
FOLDER_PATH = "test/checkpoint-1500"  # 업로드할 로컬 폴더 경로
REPO_ID = "Dongwookss/vie_task_v5"  # 본인의 username과 원하는 모델 이름으로 변경

# 업로드 실행
upload_folder_to_huggingface(
    folder_path=FOLDER_PATH,
    repo_id=REPO_ID,
    token=HF_TOKEN
)